# 03 — Business Analysis
**Input:** `exports/` — tabla maestra + pre-agregadas  
**Objetivo:** Responder las 4 preguntas de negocio con números reales

1. Ventas — tendencia mensual, categorías top, regiones
2. Operaciones — tiempos de entrega, % tardíos, peores estados
3. Clientes — ticket promedio, satisfacción, categorías con peores reviews
4. Vendedores — top performers, distribución geográfica

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

EXPORTS = '../exports/'

df           = pd.read_csv(EXPORTS + 'orders_features.csv')
agg_monthly  = pd.read_csv(EXPORTS + 'agg_monthly.csv')
agg_category = pd.read_csv(EXPORTS + 'agg_category.csv')
agg_state    = pd.read_csv(EXPORTS + 'agg_state.csv')
agg_delivery = pd.read_csv(EXPORTS + 'agg_delivery.csv')
agg_sellers  = pd.read_csv(EXPORTS + 'agg_sellers.csv')
agg_payment  = pd.read_csv(EXPORTS + 'agg_payment.csv')

print(f'Tabla maestra: {df.shape}')
print(f'Revenue total: R$ {df["revenue"].sum():,.0f}')

---
## 1. VENTAS

In [ ]:
# --- 1A. Tendencia mensual de revenue ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Revenue por mes
axes[0].plot(agg_monthly['purchase_year_month'], agg_monthly['revenue'],
             color='steelblue', marker='o', markersize=4, linewidth=2)
axes[0].set_title('Monthly Revenue Trend (Sep 2016 – Oct 2018)')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=45)

# Órdenes por mes
axes[1].bar(agg_monthly['purchase_year_month'], agg_monthly['orders'],
            color='steelblue', alpha=0.7)
axes[1].set_title('Monthly Orders Count')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

best_month = agg_monthly.loc[agg_monthly['revenue'].idxmax()]
print(f'Mejor mes: {best_month["purchase_year_month"]} — R$ {best_month["revenue"]:,.0f} | {best_month["orders"]:,.0f} ordenes')

In [ ]:
# --- 1B. Top 10 categorías por revenue ---
top10 = agg_category.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top10['product_category_name_english'][::-1], top10['revenue'][::-1], color='steelblue')
axes[0].set_title('Top 10 Categories by Revenue')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}K'))

axes[1].barh(top10['product_category_name_english'][::-1], top10['orders'][::-1], color='coral')
axes[1].set_title('Top 10 Categories by Number of Orders')

plt.tight_layout()
plt.show()

top3_rev = agg_category.head(3)[['product_category_name_english','revenue','orders']]
print('Top 3 por revenue:')
print(top3_rev.to_string(index=False))

In [ ]:
# --- 1C. Revenue por estado (top 10) ---
top10_states = agg_state.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top10_states['customer_state'][::-1], top10_states['revenue'][::-1], color='steelblue')
axes[0].set_title('Top 10 States by Revenue')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}K'))

axes[1].barh(top10_states['customer_state'][::-1], top10_states['customers'][::-1], color='mediumseagreen')
axes[1].set_title('Top 10 States by Unique Customers')

plt.tight_layout()
plt.show()

sp_pct = agg_state[agg_state['customer_state']=='SP']['revenue'].values[0] / agg_state['revenue'].sum()
print(f'SP concentra el {sp_pct:.1%} del revenue total')

---
## 2. OPERACIONES

In [ ]:
# --- 2A. Distribución de tiempos de entrega ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['delivery_days'].clip(0, 60).hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(df['delivery_days'].median(), color='red', linestyle='--',
                label=f'Median: {df["delivery_days"].median():.0f} days')
axes[0].axvline(df['delivery_days'].mean(), color='orange', linestyle='--',
                label=f'Mean: {df["delivery_days"].mean():.1f} days')
axes[0].set_title('Delivery Time Distribution (days, capped at 60)')
axes[0].legend()

# Late vs on time
late_counts = df.drop_duplicates('order_id')['is_late'].value_counts().rename({0:'On Time', 1:'Late'})
axes[1].bar(late_counts.index, late_counts.values, color=['mediumseagreen','lightcoral'])
axes[1].set_title('On-Time vs Late Deliveries')
for i, v in enumerate(late_counts.values):
    axes[1].text(i, v + 300, f'{v:,}\n({v/late_counts.sum():.1%})', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f'Avg delivery: {df["delivery_days"].mean():.1f} days | Median: {df["delivery_days"].median():.0f} days')
print(f'Late orders: {df["is_late"].mean():.1%}')

In [ ]:
# --- 2B. Estados con peor performance de entrega ---
worst_states = agg_state.sort_values('pct_late', ascending=False).head(10)
best_states  = agg_state.sort_values('avg_delivery_days').head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(worst_states['customer_state'][::-1],
             worst_states['pct_late'][::-1] * 100, color='lightcoral')
axes[0].set_title('States with Highest % Late Deliveries')
axes[0].set_xlabel('% Late')

axes[1].barh(best_states['customer_state'][::-1],
             best_states['avg_delivery_days'][::-1], color='steelblue')
axes[1].set_title('States with Fastest Average Delivery')
axes[1].set_xlabel('Avg days')

plt.tight_layout()
plt.show()

print('Peores estados (% tardios):')
print(worst_states[['customer_state','pct_late','avg_delivery_days','orders']].head(5).to_string(index=False))

---
## 3. CLIENTES

In [ ]:
# --- 3A. Ticket promedio y distribución ---
order_totals = df.groupby('order_id')['revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order_totals.clip(0, 500).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(order_totals.median(), color='red', linestyle='--',
                label=f'Median: R${order_totals.median():.0f}')
axes[0].axvline(order_totals.mean(), color='orange', linestyle='--',
                label=f'Mean: R${order_totals.mean():.0f}')
axes[0].set_title('Order Value Distribution (capped R$500)')
axes[0].set_xlabel('Order Value (R$)')
axes[0].legend()

# Review score distribution
df.drop_duplicates('order_id')['review_score'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='steelblue', edgecolor='white'
)
axes[1].set_title('Review Score Distribution')
axes[1].set_xlabel('Score')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f'Avg ticket: R${order_totals.mean():.2f} | Median: R${order_totals.median():.2f}')
print(f'Avg review score: {df["review_score"].mean():.2f}')

In [ ]:
# --- 3B. Categorías con peores reviews ---
# Solo categorías con al menos 100 reviews para que sea representativo
cat_reviews = agg_category[agg_category['orders'] >= 100].sort_values('avg_review')
worst_reviews = cat_reviews.head(10)
best_reviews  = cat_reviews.sort_values('avg_review', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(worst_reviews['product_category_name_english'][::-1],
             worst_reviews['avg_review'][::-1], color='lightcoral')
axes[0].set_title('Categories with Lowest Avg Review Score (min 100 orders)')
axes[0].set_xlim(0, 5)
axes[0].axvline(4.09, color='gray', linestyle='--', label='Overall avg (4.09)')
axes[0].legend()

axes[1].barh(best_reviews['product_category_name_english'][::-1],
             best_reviews['avg_review'][::-1], color='mediumseagreen')
axes[1].set_title('Categories with Highest Avg Review Score')
axes[1].set_xlim(0, 5)
axes[1].axvline(4.09, color='gray', linestyle='--', label='Overall avg (4.09)')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Peores categorias por review:')
print(worst_reviews[['product_category_name_english','avg_review','orders']].to_string(index=False))

---
## 4. VENDEDORES

In [ ]:
# --- 4A. Top 15 sellers por revenue ---
top15_sellers = agg_sellers.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(range(len(top15_sellers)), top15_sellers['revenue'][::-1].values, color='steelblue')
axes[0].set_yticks(range(len(top15_sellers)))
axes[0].set_yticklabels([s[:8]+'...' for s in top15_sellers['seller_id'][::-1]])
axes[0].set_title('Top 15 Sellers by Revenue')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}K'))

# Sellers por estado
seller_state_dist = agg_sellers['seller_state'].value_counts().head(10)
axes[1].bar(seller_state_dist.index, seller_state_dist.values, color='steelblue')
axes[1].set_title('Seller Distribution by State')

plt.tight_layout()
plt.show()

print(f'Total sellers activos: {len(agg_sellers):,}')
top_seller = agg_sellers.iloc[0]
print(f'Top seller: R$ {top_seller["revenue"]:,.0f} | {top_seller["orders"]} ordenes | Estado: {top_seller["seller_state"]}')
# Concentracion: top 10% de sellers = % del revenue
top10pct = agg_sellers.head(int(len(agg_sellers)*0.1))
print(f'Top 10% sellers concentran: {top10pct["revenue"].sum()/agg_sellers["revenue"].sum():.1%} del revenue')

---
## 5. KEY FINDINGS — resumen ejecutivo

In [ ]:
best_month     = agg_monthly.loc[agg_monthly['revenue'].idxmax()]
top_cat        = agg_category.iloc[0]
sp_pct         = agg_state[agg_state['customer_state']=='SP']['revenue'].values[0] / agg_state['revenue'].sum()
order_totals   = df.groupby('order_id')['revenue'].sum()
worst_state    = agg_state.sort_values('pct_late', ascending=False).iloc[0]
worst_cat_rev  = agg_category[agg_category['orders']>=100].sort_values('avg_review').iloc[0]
top10pct_rev   = agg_sellers.head(int(len(agg_sellers)*0.1))['revenue'].sum() / agg_sellers['revenue'].sum()

print(f'''
KEY FINDINGS
============

VENTAS
- Revenue total: R$ {df['revenue'].sum():,.0f} | {df['order_id'].nunique():,} orders
- Peak month: {best_month['purchase_year_month']} — R$ {best_month['revenue']:,.0f}
- Top category: {top_cat['product_category_name_english']} (R$ {top_cat['revenue']:,.0f})
- Sao Paulo = {sp_pct:.1%} of total revenue

OPERATIONS
- Avg delivery time: {df['delivery_days'].mean():.1f} days (median {df['delivery_days'].median():.0f} days)
- On-time rate: {1 - df['is_late'].mean():.1%} ({df['is_late'].mean():.1%} late)
- Worst state for late deliveries: {worst_state['customer_state']} ({worst_state['pct_late']:.1%} late)

CUSTOMERS
- Avg order value: R$ {order_totals.mean():.2f} (median R$ {order_totals.median():.2f})
- Avg review score: {df['review_score'].mean():.2f} / 5.0
- Lowest-rated category: {worst_cat_rev['product_category_name_english']} ({worst_cat_rev['avg_review']:.2f})

SELLERS
- Active sellers: {len(agg_sellers):,}
- Top 10% sellers = {top10pct_rev:.1%} of revenue (concentration)
''')